# Input Parameters

In [ ]:
%sql
CREATE WIDGET TEXT catalog_source DEFAULT 'silver_dev';
CREATE WIDGET TEXT catalog_sink DEFAULT 'gold_dev';

DECLARE OR REPLACE VARIABLE schema_source STRING DEFAULT 'reservations';
DECLARE OR REPLACE VARIABLE schema_sink STRING DEFAULT 'reservations_analytics';

# Create Schema Sink

In [ ]:
%sql 
CREATE SCHEMA IF NOT EXISTS IDENTIFIER(:catalog_sink || '.' || schema_sink)

# Load tables

In [ ]:
%sql
CREATE OR REPLACE TABLE IDENTIFIER(:catalog_sink || '.' || schema_sink || '.obt_payments_agg') CLUSTER BY AUTO
COMMENT 'Tabela OBT(One Big Table) agregada que consolida informações de reservas, usuários e pagamentos, permitindo análises detalhadas de comportamento de clientes, status de reservas e métodos de pagamento.'
TBLPROPERTIES (
  'data.quality.level' = 'Gold',
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact' = 'true'
)
AS
SELECT
  u.country,
  u.continent,
  u.user_type,
  u.is_business,
  u.company_name,
  b.status AS booking_status, 
  b.check_in, 
  b.check_out, 
  p.payment_method, 
  p.status AS payment_status,  
  SUM(b.guests_count) AS total_guests, 
  SUM(b.total_amount) AS total_booking_amount, 
  SUM(p.amount) AS total_payment_amount
FROM 
  IDENTIFIER(:catalog_source || '.' || schema_source || '.users') AS u 
  JOIN IDENTIFIER(:catalog_source || '.' || schema_source || '.bookings') AS b  
  ON u.user_id = b.user_id
  JOIN IDENTIFIER(:catalog_source || '.' || schema_source || '.payments') AS p  
  ON b.booking_id = p.booking_id
GROUP BY
  ALL